# Production Deployment with Slurm

This notebook covers best practices for deploying Large Language Models (LLMs) in production on HPC clusters using Slurm. You'll learn how to:

- Write Slurm job scripts for LLM inference and training
- Deploy FastAPI/Flask services for model serving
- Set up monitoring and logging for production
- Implement horizontal scaling and load balancing

---

## 1. Slurm Job Scripts for LLM Inference and Training

Below are example Slurm scripts for both inference (serving) and training jobs.

### 1.1 Inference Job Script Example

```bash
#!/bin/bash
#SBATCH --job-name=llm-inference
#SBATCH --partition=gpu
#SBATCH --gres=gpu:1
#SBATCH --cpus-per-task=4
#SBATCH --mem=32G
#SBATCH --time=02:00:00
#SBATCH --output=logs/inference_%j.log

module load Miniforge3 cuda/12.1
source activate llm-prod

export TRANSFORMERS_CACHE=$HOME/.cache/huggingface
export HF_HOME=$HOME/.cache/huggingface

# Start FastAPI server
uvicorn serve_llm:app --host 0.0.0.0 --port 8000
```

### 1.2 Training Job Script Example

```bash
#!/bin/bash
#SBATCH --job-name=llm-train
#SBATCH --partition=gpu
#SBATCH --gres=gpu:4
#SBATCH --cpus-per-task=16
#SBATCH --mem=128G
#SBATCH --time=24:00:00
#SBATCH --output=logs/train_%j.log

module load Miniforge3 cuda/12.1
source activate llm-prod

export TRANSFORMERS_CACHE=$HOME/.cache/huggingface
export HF_HOME=$HOME/.cache/huggingface

# Distributed training with accelerate
accelerate launch train_llm.py --config configs/train_config.yaml
```

## 2. FastAPI/Flask Service Deployment Example

Below is a minimal FastAPI app for serving your fine-tuned LLM.

In [ ]:
# serve_llm.py
from fastapi import FastAPI, Request
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import torch

app = FastAPI()

MODEL_NAME = "microsoft/phi-2"
LORA_PATH = "./deployment_model/lora_adapter"
TOKENIZER_PATH = "./deployment_model/tokenizer"

tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_PATH)
base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float16, device_map="auto")
model = PeftModel.from_pretrained(base_model, LORA_PATH)

@app.post("/generate")
async def generate(request: Request):
    data = await request.json()
    prompt = data.get("prompt", "")
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=128, temperature=0.7)
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return {"response": response[len(prompt):].strip()}

### Flask Alternative

Replace FastAPI imports with Flask and use `@app.route('/generate', methods=['POST'])` for similar logic.

## 3. Monitoring and Logging Setup

- **Slurm logs**: Use `#SBATCH --output=logs/job_%j.log` for job logs.
- **Application logs**: Use Python's `logging` module or tools like [loguru](https://github.com/Delgan/loguru).
- **Prometheus/Grafana**: For metrics, run a Prometheus node exporter and use [prometheus-fastapi-instrumentator](https://github.com/trallnag/prometheus-fastapi-instrumentator).

### Example: FastAPI Logging
```python
import logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)
logger.info("LLM service started")
```

### Example: Prometheus Metrics
```python
from prometheus_fastapi_instrumentator import Instrumentator
Instrumentator().instrument(app).expose(app)
```

## 4. Horizontal Scaling and Load Balancing

### 4.1 Multiple Inference Workers
- Launch multiple inference jobs on different nodes/GPUs using Slurm job arrays or separate jobs.

### 4.2 Load Balancer Example (NGINX)
```nginx
http {
    upstream llm_backend {
        server node01:8000;
        server node02:8000;
        server node03:8000;
    }
    server {
        listen 80;
        location / {
            proxy_pass http://llm_backend;
        }
    }
}
```

### 4.3 Slurm Job Array for Inference
```bash
#SBATCH --array=0-3
# Each task runs an inference server on a different node/GPU
```

### 4.4 Service Discovery
- Use a service registry (e.g., Consul) or static host lists for NGINX upstreams.
- For dynamic scaling, consider Kubernetes on HPC (if available).

---
## Summary

- Use Slurm scripts for robust, reproducible LLM deployment
- Serve models with FastAPI/Flask for REST APIs
- Monitor with Slurm logs, Python logging, and Prometheus
- Scale horizontally with job arrays and load balancers

**Next steps:**
- Integrate authentication and request throttling
- Automate deployment with CI/CD pipelines
- Explore GPU utilization metrics for cost optimization


© mattbixley 2025